Gradient Boosting Classifier

Part 1: Predicting if a Storm Would Occur using Gradient Boosting Classifier

Model:
    Objective: 
        min Z1 classification error, when predicting tropical storms
    Constraints:
        CO2 emisions
        Monthly temps (Jan to Dec)
        Year
        Month,
        Day

        Month should between Jan-Dec
            If month Jan, Mar, May, Jul, Aug, Oct, Dec
	            day>= 1 && day<=31
            If month Apr, Jun, Sep, Nov
	            day>= 1 && day<=30
            If month Feb
	            day>=1 && day <=28
            If year %4 && year %100 &year %400
		        day>= 1 && day<=29	

Expected output - binary value 1 for tropical storm occurred else 0

In [38]:
#imports
import pandas as pd
import numpy as np
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.ensemble import GradientBoostingClassifier
import time
from sklearn.metrics import accuracy_score, precision_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

In [2]:
#Reading from the dataset
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

#testing if the reading from the dataset was successful
print(data.head(5))

   Year  MONTH  DAY   LAT  LONG  WIND_KTS  PRESSURE CAT  Shape_Leng Country  \
0  1880      8   11  23.0 -91.9        70         0  H1    0.806226  Mexico   
1  1880      8   11  23.4 -92.6        80         0  H1    0.761577  Mexico   
2  1880      8   11  23.7 -93.3        80         0  H1    0.583095  Mexico   
3  1880      8   12  24.0 -93.8        90         0  H2    0.670820  Mexico   
4  1880      9    6  23.9 -88.6        40         0  TS    0.360555  Mexico   

   ...   Jun   Jul   Aug   Sep   Oct   Nov   Dec  Storm Intensity  \
0  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         56.43582   
1  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.92616   
2  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         46.64760   
3  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.37380   
4  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         14.42220   

   Storm Intensity Label  Wind Speed Squared  
0                      3                4900  
1               

In [3]:
def valid_date(row):
    month = row ["MONTH"]
    day = row ["DAY"]
    year = row["Year"]

    if month < 1 or month > 12:
        return False
    if month in [1,3,5,7,8,10,12] and not (1<=day <= 31):
        return False
    if month in [4,6,9,11] and not (1<= day <=30):
        return False
    if month == 2:
        leap_year = (year %4 == 0 and year % 100 != 0) or (year %400 == 0)
        possible_day = 29 if leap_year else 28

        if not (1 <= day <= possible_day):
            return False
    return True
    
    #removing in valid dates from the dataset 
    #data = data[data.apply(valid_date, axis=1)]

Since our dataset only has data when tropical storms occured inorder to use the classifier to train the models, we'll have to use "dummy values" in order to train the model for the classifier to learn te difference. 

Generating realistic "dummy values" by  using the dates what storms did not occur and initilizing information about the storms to be 0 based on their data types

In [4]:
data["storm_occured"] = 1

#possible dates
years = range(data["Year"].min(), data["Year"].max())
months = range(1,12)
days = range (1, 31)

#possible locations, 20.0 away from the actural tropical storm location
lats =  np.arange(data["LAT"].min(), data["LAT"].max(), 20.0)
longs = np.arange(data["LONG"].min(), data["LONG"].max(), 20.0)

new_rows = pd.DataFrame(product(years, months, days, lats, longs), columns= ["Year", "MONTH", "DAY", "LAT", "LONG"])

#only keeping the rows with valid dates from the new_rows
new_rows = new_rows[new_rows.apply(valid_date, axis=1)]

#Dropping the dates where a storm actually occured
storm_info = data[["Year", "MONTH", "DAY", "LAT", "LONG"]].drop_duplicates()

#comparing the 2 datasets to find the date that is in the new_rows and not in the cleaned tropical storm dataset
merge_data = pd.merge(new_rows, storm_info, how = "left", on= ["Year", "MONTH", "DAY", "LAT", "LONG"], indicator= "merge")
no_storms = merge_data[merge_data["merge"]== "left_only"].drop(columns=["merge"])

#Adding in initilizing values to the no_storm rows
no_storms["storm_occured"] = 0
no_storms["WIND_KTS"] = 0
no_storms["CAT"] = "NA"
no_storms["Storm Intensity"] = 0.0
no_storms["Storm Intensity Label"] = 0

#Using the avg golbal temps and CO2 levels for the no_storms
for column in ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec", "CO2 emission (Tons)"]:
    if column in data.columns:
        no_storms[column] = data[column].mean()

#joing the no_storms to the dataset with the storms
join_data = pd.concat([data, no_storms], ignore_index= True)

#Returning all the rows in a random order
shuffle_data = join_data.sample(frac=1).reset_index(drop=True)

#Updating the csv with the no_storms
shuffle_data.to_csv("completed_dataset_for_IS_project_25.csv", index= False)

Checking if the no storm data was added into the dataset

In [5]:
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

no_storm = data[data["storm_occured"]== 0]
yes_storm = data[data["storm_occured"]== 1]

print(no_storm.shape, yes_storm.shape)

(3025079, 27) (52656, 27)


The data is unbalanced leaning to the no storm dat which would cause our model to have a bias to no storm, will have to balce the data for a better prediction 

In [ ]:
no_storm = no_storm.head(yes_storm.shape[0])
print(no_storm.shape, yes_storm.shape)

(52656, 27) (52656, 27)


Joining and shuffling the remaining data

In [9]:
data = pd.concat([no_storm, yes_storm])
data = shuffle(data)
data.head(5)

,Year,MONTH,DAY,LAT,LONG,WIND_KTS,PRESSURE,CAT,Shape_Leng,Country,...,Jul,Aug,Sep,Oct,Nov,Dec,Storm Intensity,Storm Intensity Label,Wind Speed Squared,storm_occured
1326352,1993,9,16,12.5,-84.9,30,1006.0,TD,0.640312,Nicaragua,...,0.250000,0.110000,0.120000,0.230000,0.030000,0.180000,19.20936,1,900.0,1
2688361,1972,8,16,13.3,-157.4,90,0.0,H2,0.707107,United States,...,0.010000,0.160000,0.020000,0.080000,0.020000,0.180000,63.63963,4,8100.0,1
34231,1947,3,29,44.2,-20.0,0,NaN,NaN,NaN,NaN,...,0.102566,0.097883,0.099385,0.115567,0.109884,0.100033,0.00000,0,NaN,0
21726,1923,9,1,64.2,0.0,0,NaN,NaN,NaN,NaN,...,0.102566,0.097883,0.099385,0.115567,0.109884,0.100033,0.00000,0,NaN,0
12835,1974,6,7,24.2,40.0,0,NaN,NaN,NaN,NaN,...,0.102566,0.097883,0.099385,0.115567,0.109884,0.100033,0.00000,0,NaN,0


Training the model

In [ ]:
features = ["Year", "MONTH", "DAY", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
target = "storm_occured"

X= data[features]
y = data[target]

#Spliting the merged dataset into training (70%), validation (15%), and testing (15%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=5, stratify=y) #for balance split
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=5, stratify=y_temp)

#testing splits
print(f"Total size: {len(X)}")
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")


Total size: 105312
Train size: 73718
Validation size: 15797
Test size: 15797


Training the Model

In [14]:
#dropping rows with missing values
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

X_val = X_val.dropna()
y_val = y_val.loc[X_val.index]

print(y_train.value_counts())

model = GradientBoostingClassifier(random_state=5)
model.fit(X_train, y_train)

storm_occured
1    36859
0    36859
Name: count, dtype: int64


GradientBoostingClassifier(random_state=5)

Evaluation using the test and validation data

Classification → Accuracy, Precision
Efficiency → Latency


In [ ]:
#Test
start_time_test = time.time()
y_test_prediction = model.predict(X_test)
latency_test = time.time() - start_time_test

accuracy_test = accuracy_score(y_test, y_test_prediction)
precision_test = precision_score(y_test, y_test_prediction)

print("Evaluation Test")
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Latency: {latency_test}")

Evaluation Test
Accuracy: 1.0
Precision: 1.0
Latency: 0.06612920761108398


For the result of:
Accuracy: 1.0
Precision: 1.0

I will assume that is is because that data for storm and no storm are very separable/very distint

In [18]:
#Validation
start_time_val = time.time()
y_val_prediction = model.predict(X_val)
latency_val = time.time() - start_time_val

accuracy_val = accuracy_score(y_val, y_val_prediction)
precision_val = precision_score(y_val, y_val_prediction)

print("Evaluation Validation")
print(f"Accuracy: {accuracy_val}")
print(f"Precision: {precision_val}")
print(f"Latency: {latency_val}")

Evaluation Validation
Accuracy: 1.0
Precision: 1.0
Latency: 0.03983926773071289


For the user to add in their variables to predict if a storm would occur or not and if a storm does occur it would lead the user into the other model to predict the intensity

In [24]:
def predict_storm(model, year, month, day, co2, temps_dict):
    input = {
        "Year": [year],
        "MONTH": [month],
        "DAY": [day],
        "CO2 emission (Tons)": [co2],
    }

    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
    
    for month_temp in months:
        input[month_temp] = [temps_dict.get(month_temp, 0)]

    input_data = pd.DataFrame(input)
    prediction = model.predict(input_data)[0]

    return prediction #0 or 1 for the model in part 2


In [ ]:
'''
#example for predicting a possible storm
possible_temps= {"Jan": 27.0, "Feb": 27.3, "Mar": 27.8, "Apr": 28.1, "May": 28.6, "Jun": 29.0, "Jul": 29.3, "Aug": 29.4, "Sep": 29.2, "Oct": 28.7, "Nov": 28.1, "Dec": 27.5}
will_storm_occur = predict_storm(model=model, year=2024, month=8, day=14, co2= 412.5, temps_dict = possible_temps)
print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")
'''

A storm was predicted


Since that original data did not have any data for no storms, and I had to make up the data using valid dates and lat and longs where it had no recorded storms and average global temps and CO2 levels, it might not be the most accurate in predicting no storm events. 

For this example for me to predict a result of no storm I'll have to calculate the average global temps and CO2 levels

In [59]:
#average global temps and CO2 levels
avg_column = ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec", "CO2 emission (Tons)"]
avg = {col: data[col].mean() for col in avg_column if col in data.columns}
 
possible_temps= {month: avg[month] for month in ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]}
avg_co2 = avg["CO2 emission (Tons)"]

In [ ]:
'''
#example for predicting a no storm
will_storm_occur = predict_storm(model=model, year=2024, month=8, day=14, co2= avg_co2, temps_dict = possible_temps)
print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")
'''

No storm was predicted


Logistic Regression

Part 2: Predicting the intensity of the storm

Objective: 
    min Z2 error in predicting the intensity of the tropical storm
Constraints:
	    Wind_kts, pressure, category >= 0 
        stormed_occured =1
	    Storm Intensity Label >= 1 && Storm Intensity Label <=5


In [52]:
#Only accessing the rows were a storm occurred to train the models on the intensity of the tropical storms

storms = data[(data["storm_occured"] ==1) & (data["Storm Intensity Label"] >=1) &(data["Storm Intensity Label"] <=5)].copy()

In [53]:
#Converting the CAT values to int for machine learning
le = LabelEncoder()
storms["CAT_int"] = le.fit_transform(storms["CAT"].astype(str))

Training the model

In [54]:
features_2 = storms[["CAT_int", "Year", "MONTH", "DAY", "CO2 emission (Tons)", "WIND_KTS"]]
target_2 = storms["Storm Intensity Label"]

X= features_2
y= target_2

model_2 = LogisticRegression(multi_class= "multinomial", max_iter=1000)
model_2.fit(X, y)

C:\Users\katoy\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=1000, multi_class='multinomial')

For the user to add in their variables to predict the insensity of a storm 

In [55]:
def predict_intensity(model, wind_kts, cat, co2, year, month, day):
    cat_int = le.transform([cat])[0]
    input = pd.DataFrame([{
    "CAT_int": cat_int,
    "Year": year,
    "MONTH":month,
    "DAY": day,
    "CO2 emission (Tons)": co2,
    "WIND_KTS": wind_kts
    }])

    return model.predict(input)[0]

In [ ]:
#example for predicting a possible storm
possible_temps= {"Jan": 27.0, "Feb": 27.3, "Mar": 27.8, "Apr": 28.1, "May": 28.6, "Jun": 29.0, "Jul": 29.3, "Aug": 29.4, "Sep": 29.2, "Oct": 28.7, "Nov": 28.1, "Dec": 27.5}
will_storm_occur = predict_storm(model=model, year=2024, month=8, day=14, co2= 412.5, temps_dict = possible_temps)


#example for predicting a no storm
#will_storm_occur = predict_storm(model=model, year=2024, month=8, day=14, co2= avg_co2, temps_dict = possible_temps)


if will_storm_occur== 1:
    print("A storm was predicted, now predicting it's intensity...")
    intensity = predict_intensity(model = model_2, wind_kts=70, cat= "H2", co2=412.5, year=2024, month=8, day=14)
    #print(f"Predicted storm intensity: {intensity}")
    print("Based on MyNASEData Hurricane Dynamics")
    
    if intensity == 1:
        value = "Minimal"
    elif intensity == 2:
        value = "Moderate"
    elif intensity == 3:
        value = "Extensive"
    elif intensity == 4:
        value = "Extreme"
    elif intensity == 5:
        value = "Catastrophic"
    else:
        value = "Unknown"

    print(f"Predicted storm intensity: {value}")
else:
    print(f"No storm was predicted, skipping the intensity prediction")

A storm was predicted, now predicting it's intensity...
Based on MyNASEData Hurricane Dynamics
Predicted storm intensity: Moderate
